In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip
from qutip.measurement import measurement_statistics_observable

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track

# Linalg libraries
import numpy as np
import h5py as hf

# Helper libraries
from dataclasses import dataclass

In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
background = np.genfromtxt("data/background.csv", delimiter=",", skip_header=1)[:, 1:][:5000]
signal = np.genfromtxt("data/signal.csv", delimiter=",", skip_header=1)[:, 1:][:5000]

In [ ]:
background = (background - background.mean(axis=0)) / background.std(axis=0)
signal = (signal - signal.mean(axis=0)) / signal.std(axis=0)

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length
    initial_state = []
    for i in range(number_of_spins):
        initial_state.append(
            basis(2, np.random.randint(low=0, high=2, size=1)[0])
        )

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]
    
    # Hamiltonian - Energy splitting terms
    H = 0
    H -= driving_field[t][0] * sz_list[0]
    H -= driving_field[t][1] * sz_list[5]

    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]
    
    return H
    

In [ ]:
simulation_parameters = SimulationParameters(
    length=10,
    coupling=2.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": background
}

In [ ]:
# Observables for state description.
state_dimension = 100

measurements = []

for _ in range(state_dimension):
    seed = np.random.randint(641)
    measurements.append(
        qutip.rand_herm(
            16,
            0.5, 
            dims=[[2, 2, 2, 2], [2, 2, 2, 2]]
        )
    )

In [ ]:
signal_length = background.shape[0]

# Equilibration run
hamiltonian = compute_hamiltonian(0, args)
times = np.linspace(0, 1, 5)
new_state = mesolve(hamiltonian, simulation_state.quantum_state, times, [], [], args).states[-1]

# Real simulation
observations = np.zeros((signal_length, state_dimension))
for t in track(range(signal_length), description="Running Simulation"):
    hamiltonian = compute_hamiltonian(t, args)
    new_state = mesolve(hamiltonian, new_state, times, [], [], args).states[-1]
    for k, operator in enumerate(measurements):
        measure_state = new_state.ptrace((3, 4, 6, 7))
        observations[t][k] = qutip.expect(measure_state * measure_state.dag(), operator)

np.save("background_observables.npy", observations)

In [ ]:
simulation_parameters = SimulationParameters(
    length=10,
    coupling=2.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": signal
}

In [ ]:
signal_length = signal.shape[0]

# Equilibration run
hamiltonian = compute_hamiltonian(0, args)
times = np.linspace(0, 0.1, 10)
new_state = mesolve(hamiltonian, simulation_state.quantum_state, times, [], [], args).states[-1]

# Real simulation
observations = np.zeros((signal_length, state_dimension))
for t in track(range(signal_length), description="Running Simulation"):
    hamiltonian = compute_hamiltonian(t, args)
    new_state = mesolve(hamiltonian, new_state, times, [], [], args).states[-1]
    for k, operator in enumerate(measurements):
        measure_state = new_state.ptrace((3, 4, 6, 7))
        observations[t][k] = qutip.expect(measure_state * measure_state.dag(), operator)

np.save("signal_observables.npy", observations)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()
        
        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 500),
            nn.ReLU(),
            nn.Linear(500, output_dimension)
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data).to(device)
        self.function_data = torch.Tensor(function_data).to(device)
        
        self.norm_factor = 1 # max(abs(function_data.flatten()))
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: Avg loss: {test_loss:>8f} \n")
    
    return test_loss

In [ ]:
def train_model(state_data, function_data, model = None):
    """
    Train a model on the current data.
    """

    dataset = ReservoirDataset(
        state_data=state_data,
        function_data=function_data,
    )
    
    if model is None:
        model = NeuralNetwork(
            state_dimension=100, 
            output_dimension=2
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Create the loader
    loader = DataLoader(dataset, batch_size=150, shuffle=False)
    
    # Train the network
    epochs = 100000
    train_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)

    train_losses = [item.cpu().detach().numpy() for item in train_losses]
    
    return train_losses, model

In [ ]:
background_ds = np.load("background_observables.npy", allow_pickle=True)
signal_ds = np.load("signal_observables.npy", allow_pickle=True)

train_indices = np.random.randint(0, len(background) - 1, size=(7000,))
train_features = np.take(background_ds, train_indices, axis=0)
train_targets = np.take(background, train_indices, axis=0)

losses, model = train_model(background_ds, background, model=model)

In [ ]:
plt.plot(losses)

plt.yscale("log")

In [ ]:
background_predictions = model(torch.Tensor(background_ds).to(device)).cpu().detach().numpy()
signal_predictions = model(torch.Tensor(signal_ds).to(device)).cpu().detach().numpy()

In [ ]:
fig, ax = plt.subplots(1, 2)

for i in range(2):
    ax[i].plot(background[:, i], background_predictions[:, i], 'r.')
    ax[i].plot(background[:, i], background[:, i], 'k--')

plt.show()